In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
from sklearn.model_selection import TimeSeriesSplit, StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, make_scorer, fbeta_score, precision_score, recall_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
chunk_size = 1_000_000

files = {
    "Train": "train.csv",
    "Validation": "validation.csv",
    "Test": "test.csv"
}

for split_name, file_path in files.items():
    print(f"\n{split_name} chunks:")

    for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size), start=1):
        print(f"{split_name} chunk {i}: {chunk.shape}")

In [ ]:

X_train = train_df.drop(columns=[y])
y_train = train_df[y]
    
X_val = val_df.drop(columns=[y])
y_val = val_df[y]

X_test = test_df.drop(columns=[y])
y_test = test_df[y]

In [ ]:
# Define features and target variable
X = data[[
    "Stop:Station code", 
    "Hour",
    "DayOfWeek_sin",
    "DayOfWeek_cos", 
    "Month_sin", 
    "Month_cos", 
    "IsWeekend", 
    "RushHour",
    "StationTraffic",
    "Temperature",
    "Humidity",
    "Air Pressure",
    "Horizontal Visibility",
    "Cloud Cover",
    "Precipitation Amount",
    "Precipitation Duration",
    "Hourly Mean Wind Speed",
    "Max Wind Speed",
    "Fog",
    "Rainfall",
    "Snowfall",
    "Thunder",
    "Hail",
    "Service:Type"
    ]]

y = data["is_cancelled"]

In [ ]:
X = pd.get_dummies(
    X,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)

# Baseline: Logistic Regression

In [ ]:
# # # # 9M49.0S runtime

# logist = LogisticRegression(
#     max_iter=1000,
#     class_weight="balanced",
#     solver="liblinear"
# )

# logist.fit(X, y)
# y_pred = logist.predict(X)

# print(classification_report(y, y_pred))

# # ## precision recall f1-score support 0 0.95 0.51 0.67 65145269 1 0.06 0.55 0.12 3972526 
# # # accuracy 0.51 69117795 
# # # macro avg 0.51 0.53 0.39 69117795 
# # # weighted avg 0.90 0.51 0.63 69117795

# Random Forest
Due to the computational size of the full dataset, a stratified sampling procedure was applied prior to model training. Sampling was performed separately for each year and cancellation class to preserve both the temporal distribution and class imbalance structure of the original dataset. This approach reduced computational requirements while maintaining representative operational patterns across the analysis period.

In [ ]:
# # Keep class ratio similar to full data
# sample = comb.groupby("is_cancelled", group_keys=False).sample(
#     frac=0.15, random_state=42)

# sgkfold = StratifiedGroupKFold(
#     n_splits=5,
#     shuffle=False
# )

# y_sample = sample["is_cancelled"]
# X_sample = sample[[
#     "Hour",
#     "Temperature",
#     "Humidity",
#     "Air Pressure",
#     "Horizontal Visibility",
#     "Cloud Cover",
#     "Precipitation Amount",
#     "Precipitation Duration",
#     "Hourly Mean Wind Speed",
#     "Fog",
#     "Rainfall",
#     "Snowfall",
#     "Thunder",
#     "Hail"
# ]]

# groups_sample = sample["Service:RDT-ID"]

# rf = RandomForestClassifier(
#     n_estimators=50,
#     max_depth=8,
#     random_state=42,
#     class_weight="balanced",
#     n_jobs=-1
# )

# scoring = {
#     "precision": make_scorer(precision_score),
#     "recall": make_scorer(recall_score),
#     "f2": make_scorer(fbeta_score, beta=2)
# }

# scores = cross_validate(
#     rf,
#     X_sample,
#     y_sample,
#     cv=sgkfold,
#     groups=groups_sample,
#     scoring=scoring,
#     n_jobs=1
# )

# print("Mean Precision:", scores["test_precision"].mean())
# print("Mean Recall:", scores["test_recall"].mean())
# print("Mean F2:", scores["test_f2"].mean())

In [ ]:
# rf.fit(X_sample, y_sample)

# importances = pd.Series(
#     rf.feature_importances_,
#     index=X_sample.columns
# ).sort_values(ascending=False)

# print(importances)

In [ ]:
# # Define features for the full dataset training
# features = [
#     "Hour",
#     "Temperature",
#     "Humidity",
#     "Air Pressure",
#     "Horizontal Visibility",
#     "Cloud Cover",
#     "Precipitation Amount",
#     "Precipitation Duration",
#     "Hourly Mean Wind Speed",
#     "Fog",
#     "Rainfall",
#     "Snowfall",
#     "Thunder",
#     "Hail"
# ]

# # Sort the combined dataset by date to ensure temporal order for training
# comb = comb.sort_values("Service:Date")

# classes = np.array([0, 1])

# # Compute class weights to handle imbalance in the target variable
# weights = compute_class_weight(
#     class_weight="balanced",
#     classes=classes,
#     y=comb["is_cancelled"]
# )

# # Map class weights to the corresponding classes
# class_weights = {
#     0: weights[0],
#     1: weights[1]
# }

# print("Class weights:", class_weights)

# # Define chunk size for incremental training 
# chunk_size = 2_000_000

# chunks = [
#     comb.iloc[i:i + chunk_size]
#     for i in range(0, len(comb), chunk_size)
# ]

# print(f"Total chunks: {len(chunks)}")

# # Initialize Random Forest with warm_start=True to allow incremental training
# rf = RandomForestClassifier(
#     warm_start=True,
#     n_estimators=10,
#     max_depth=8,
#     class_weight=class_weights,
#     random_state=42,
#     n_jobs=-1
# )

# # Incrementally train the model on each chunk of data
# for i, chunk in enumerate(chunks):

#     print(f"\nTraining on chunk {i+1}/{len(chunks)}")

#     X_chunk = chunk[features]
#     y_chunk = chunk["is_cancelled"]

#     rf.fit(X_chunk, y_chunk)

#     print(f"Trees trained: {rf.n_estimators}")

#     # Add additional trees after each chunk
#     rf.n_estimators += 10

# # After training on all chunks, evaluate feature importances and perform cross-validation on the sampled dataset
# importances = pd.Series(
#     rf.feature_importances_,
#     index=features
# ).sort_values(ascending=False)

# print("\nFeature Importances:")
# print(importances)

# # Define scoring metrics for cross-validation
# scoring = {
#     "precision": make_scorer(precision_score),
#     "recall": make_scorer(recall_score),
#     "f2": make_scorer(fbeta_score, beta=2)
# }

# # Perform cross-validation on the sampled dataset using the trained model
# scores = cross_validate(
#     rf,
#     X_sample,
#     y_sample,
#     cv=sgkfold,
#     groups=groups_sample,
#     scoring=scoring,
#     n_jobs=1
# )

# print("Mean Precision:", scores["test_precision"].mean())
# print("Mean Recall:", scores["test_recall"].mean())
# print("Mean F2:", scores["test_f2"].mean())

In [ ]:
# # Optimize memory usage by downcasting numerical columns
# def optimize_memory(df):
#     for col in df.select_dtypes(include=['float']):
#         df[col] = df[col].astype('float32')
#     for col in df.select_dtypes(include=['int']):
#         df[col] = df[col].astype('int32')
#     return df

# # Apply memory optimization
# comb = optimize_memory(comb)

# # Define features
# features = [
#     "Hour",
#     "Temperature",
#     "Humidity",
#     "Air Pressure",
#     "Horizontal Visibility",
#     "Cloud Cover",
#     "Precipitation Amount",
#     "Precipitation Duration",
#     "Hourly Mean Wind Speed",
#     "Fog",
#     "Rainfall",
#     "Snowfall",
#     "Thunder",
#     "Hail"
#  ]

# # Sort by date for temporal split
# comb = comb.sort_values("Service:Date").reset_index(drop=True)

# # Temporal train / validation / test split
# train_end = int(len(comb) * 0.70)
# val_end = int(len(comb) * 0.85)

# train_df = comb.iloc[:train_end]
# val_df = comb.iloc[train_end:val_end]
# test_df = comb.iloc[val_end:]

# # Optimize memory for train, validation, and test sets
# train_df = optimize_memory(train_df)
# val_df = optimize_memory(val_df)
# test_df = optimize_memory(test_df)

# X_train = train_df[features]
# y_train = train_df["is_cancelled"]

# X_val = val_df[features]
# y_val = val_df["is_cancelled"]

# X_test = test_df[features]
# y_test = test_df["is_cancelled"]

# # Compute class weights using TRAINING DATA ONLY
# classes = np.array([0, 1])

# weights = compute_class_weight(
#     class_weight="balanced",
#     classes=classes,
#     y=y_train
#  )

# class_weights = {
#     0: weights[0],
#     1: weights[1]
# }

# print("Class weights:", class_weights)

# # Train Random Forest with optimized parameters
# rf = RandomForestClassifier(
#     n_estimators=50,  # Reduce number of trees
#     max_depth=6,  # Reduce maximum depth
#     class_weight=class_weights,
#     random_state=42,
#     n_jobs=-1
#  )

# rf.fit(X_train, y_train)

# # Evaluation function
# def evaluate(model, X, y, name):
#     preds = model.predict(X)

#     precision = precision_score(y, preds, zero_division=0)
#     recall = recall_score(y, preds, zero_division=0)
#     f2 = fbeta_score(y, preds, beta=2, zero_division=0)

#     print(f"\n{name} METRICS")
#     print("Precision:", precision)
#     print("Recall:", recall)
#     print("F2:", f2)

# # Check overfitting / underfitting
# evaluate(rf, X_train, y_train, "TRAIN")
# evaluate(rf, X_val, y_val, "VALIDATION")

# # Final test evaluation only after validation check
# evaluate(rf, X_test, y_test, "TEST")

# importances = pd.Series(
#     rf.feature_importances_,
#     index=features
# ).sort_values(ascending=False)

# print("\nFeature Importances:")
# print(importances)

# XGBoost: Extreme Gradient Boosting 

In [ ]:
# scale_pos_weight = (
#     y_sample.value_counts()[0] /
#     y_sample.value_counts()[1]
# )

# xgb = XGBClassifier(
#     n_estimators=100,
#     max_depth=6,
#     learning_rate=0.1,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     scale_pos_weight=scale_pos_weight,
#     random_state=42,
#     n_jobs=-1,
#     eval_metric="logloss",
#     tree_method="hist"
# )

# f2_scorer = make_scorer(fbeta_score, beta=2)

# scores = cross_val_score(
#     xgb,
#     X_sample,
#     y_sample,
#     cv=sgkfold,
#     groups=groups_sample,
#     scoring=f2_scorer,
#     n_jobs=1
# )

# print("F2 scores:", scores)
# print("Mean F2:", scores.mean())

# xgb.fit(X_sample, y_sample)

# importances = pd.Series(
#     xgb.feature_importances_,
#     index=X_sample.columns
# ).sort_values(ascending=False)

# print(importances)

In [ ]:
# from sklearn.model_selection import cross_validate
# from sklearn.metrics import make_scorer, precision_score, recall_score, fbeta_score

# # Scorers
# precision = make_scorer(precision_score)
# recall = make_scorer(recall_score)
# f2 = make_scorer(fbeta_score, beta=2)

# scoring = {
#     "precision": precision,
#     "recall": recall,
#     "f2": f2
# }

# # Random Forest

# rf_scores = cross_validate(
#     rf,
#     X_sample,
#     y_sample,
#     cv=sgkfold,
#     groups=groups_sample,
#     scoring=scoring,
#     n_jobs=1
# )

# print("=== Random Forest ===")
# print("Mean Precision:", rf_scores["test_precision"].mean())
# print("Mean Recall:", rf_scores["test_recall"].mean())
# print("Mean F2:", rf_scores["test_f2"].mean())

# # XGBOOST

# xgb_scores = cross_validate(
#     xgb,
#     X_sample,
#     y_sample,
#     cv=sgkfold,
#     groups=groups_sample,
#     scoring=scoring,
#     n_jobs=1
# )

# print("\n=== XGBoost ===")
# print("Mean Precision:", xgb_scores["test_precision"].mean())
# print("Mean Recall:", xgb_scores["test_recall"].mean())
# print("Mean F2:", xgb_scores["test_f2"].mean())